# Activity 3: Practical Guide to Pydantic

In [1]:
!pip install pydantic

## Example 1: Your First Pydantic Model

---

**Goal**

Create a simple model for a student and learn how Pydantic checks data types and required fields.

###  Step 1: Define the schema

In [2]:
from pydantic import BaseModel

class Student(BaseModel):
  name: str
  age: int
  course: str

### Step 2: Create a student

In [3]:
student = Student(
    name="Sara",
    age=21,
    course="Data Science"
)

print(student)

name='Sara' age=21 course='Data Science'


### Step 3: Check the data types

In [4]:
print(student.name)
print(student.age)
print(type(student.age))

Sara
21
<class 'int'>


### Step 4: Try invalid data

In [5]:
from pydantic import ValidationError

try:
    student = Student(
        name="Sara",
        age="twenty-one",
        course="Data Science"
    )

except ValidationError as e:
    print("Validation failed!")
    print(e)

Validation failed!
1 validation error for Student
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty-one', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


## Example 2: Default Values and Validation Rules

### Step 1: Create the model

In [6]:
from pydantic import BaseModel, Field

class Book(BaseModel):
    title: str
    author: str
    publication_year: int = Field(ge=1900, le=2026)
    available_copies: int = Field(default=0, ge=0)

### Step 2: Create a valid book

In [7]:
book = Book(
    title="Python for Beginners",
    author="Alex Brown",
    publication_year=2024,
    available_copies=5
)

print(book)

title='Python for Beginners' author='Alex Brown' publication_year=2024 available_copies=5


### Step 3: Use the default value

In [8]:
book = Book(
    title="Learning AI",
    author="Maya Lee",
    publication_year=2025
)

print(book.available_copies)

0


### Step 4: Try invalid values

In [9]:
try:
    book = Book(
        title="Learning AI",
        author="Maya Lee",
        publication_year=1800,
        available_copies=-3
    )

except ValidationError as e:
    print("Book validation failed!")

    for error in e.errors():
        print(error)

Book validation failed!
{'type': 'greater_than_equal', 'loc': ('publication_year',), 'msg': 'Input should be greater than or equal to 1900', 'input': 1800, 'ctx': {'ge': 1900}, 'url': 'https://errors.pydantic.dev/2.13/v/greater_than_equal'}
{'type': 'greater_than_equal', 'loc': ('available_copies',), 'msg': 'Input should be greater than or equal to 0', 'input': -3, 'ctx': {'ge': 0}, 'url': 'https://errors.pydantic.dev/2.13/v/greater_than_equal'}


## Example 3: Nested Models and Lists

### Step 1: Create a model for a lesson

In [10]:
from pydantic import BaseModel

class Lesson(BaseModel):
    title: str
    duration_minutes: int

### Step 2: Create a model for the course

In [11]:
class Course(BaseModel):
    name: str
    instructor: str
    lessons: list[Lesson]

### Step 3: Validate nested data

In [13]:
course_data = {
    "name": "Python Basics",
    "instructor": "Maya Lee",
    "lessons": [
        {
            "title": "Variables and Data Types",
            "duration_minutes": 20
        },
        {
            "title": "Functions",
            "duration_minutes": 35
        }
    ]
}

In [14]:
course = Course.model_validate(course_data)

print(course)

name='Python Basics' instructor='Maya Lee' lessons=[Lesson(title='Variables and Data Types', duration_minutes=20), Lesson(title='Functions', duration_minutes=35)]


### Step 4: Access nested data

In [15]:
print(course.name)

print(course.lessons[0].title)

print(type(course.lessons[0]))

Python Basics
Variables and Data Types
<class '__main__.Lesson'>


### Step 5: Export the model

In [16]:
course_dict = course.model_dump()

print(course_dict)
print(type(course_dict))

{'name': 'Python Basics', 'instructor': 'Maya Lee', 'lessons': [{'title': 'Variables and Data Types', 'duration_minutes': 20}, {'title': 'Functions', 'duration_minutes': 35}]}
<class 'dict'>


In [17]:
course_json = course.model_dump_json(indent=2)

print(course_json)

{
  "name": "Python Basics",
  "instructor": "Maya Lee",
  "lessons": [
    {
      "title": "Variables and Data Types",
      "duration_minutes": 20
    },
    {
      "title": "Functions",
      "duration_minutes": 35
    }
  ]
}


## Example 4: Formatting Outputs from a Fine-Tuned Model

### Step 1: Define the expected output schema

In [18]:
from typing import Literal
from pydantic import BaseModel, Field

class ClassificationOutput(BaseModel):
    category: Literal["delivery", "billing", "technical", "other"]
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float = Field(ge=0, le=1)

### Step 2: Simulate a model output

In [19]:
model_output = {
    "category": "delivery",
    "sentiment": "negative",
    "confidence": 0.95
}

In [20]:
result = ClassificationOutput.model_validate(model_output)

print(result)

category='delivery' sentiment='negative' confidence=0.95


### Step 3: Access the validated fields

In [21]:
print("Category:", result.category)
print("Sentiment:", result.sentiment)
print("Confidence:", result.confidence)

Category: delivery
Sentiment: negative
Confidence: 0.95


### Step 4: Detect an invalid model output

In [22]:
bad_model_output = {
    "category": "shipping",
    "sentiment": "negative",
    "confidence": 1.5
}

try:
    result = ClassificationOutput.model_validate(bad_model_output)

except ValidationError as e:
    print("The model output is invalid!")

    for error in e.errors():
        print(error)

The model output is invalid!
{'type': 'literal_error', 'loc': ('category',), 'msg': "Input should be 'delivery', 'billing', 'technical' or 'other'", 'input': 'shipping', 'ctx': {'expected': "'delivery', 'billing', 'technical' or 'other'"}, 'url': 'https://errors.pydantic.dev/2.13/v/literal_error'}
{'type': 'less_than_equal', 'loc': ('confidence',), 'msg': 'Input should be less than or equal to 1', 'input': 1.5, 'ctx': {'le': 1.0}, 'url': 'https://errors.pydantic.dev/2.13/v/less_than_equal'}


## Example 5: Converting a Fine-Tuned Model's JSON Text into Pydantic

In [23]:
model_response = """
{
    "category": "billing",
    "sentiment": "neutral",
    "confidence": 0.88
}
"""

result = ClassificationOutput.model_validate_json(model_response)

print(result)

category='billing' sentiment='neutral' confidence=0.88


In [24]:
# Export the result
print(result.model_dump())    # This returns a Python dictionary

print(result.model_dump_json(indent=2))   # This returns a JSON string

{'category': 'billing', 'sentiment': 'neutral', 'confidence': 0.88}
{
  "category": "billing",
  "sentiment": "neutral",
  "confidence": 0.88
}
